# Notebook 1 — Read & Join the Tables

**Task 2 — From Tables to Notebooks · Qafza Tech MLOps Training 2026/2027**

**Goal:** read every Olist table on its own, understand what one row means in each one, then build a single **ML table with one row per order**.

**What we'll do:**
1. Connect to the database and read every table
2. Look at each table alone — shape, keys, duplicates
3. Aggregate `order_items` and `order_payments` up to one row per order
4. Join everything into a single order-level table
5. Sanity-check the result and save it as an artifact

> 💡 **Before you run this:** you need Task 1 done — the Postgres container running (`docker compose up -d` from the repo root) with all the Olist tables already loaded.

> ⚠️ **On purpose, we will NOT merge `olist_order_reviews` here.** Review data (and the review score) only exists once a customer has received the order and reacted to it — using it as a model input would be leakage. We keep it out of the ML table entirely, from the very first notebook.


## 1. Setup & Connection

Same connection pattern as Task 1 — secrets come from the `.env` file at the repo root, nothing hardcoded.


In [ ]:
import os
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

load_dotenv(Path("../../.env"))

DB_USER = os.getenv("POSTGRES_USER", "olist_user")
DB_PASS = os.getenv("POSTGRES_PASSWORD", "olist_pass")
DB_HOST = os.getenv("POSTGRES_HOST", "localhost")
DB_PORT = os.getenv("POSTGRES_PORT", "5432")
DB_NAME = os.getenv("POSTGRES_DB", "olist_db")

connection_string = f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string)

# Where this notebook's outputs go. Every later notebook reads from here.
ARTIFACTS_DIR = Path("artifacts/tables")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

with engine.connect() as conn:
    conn.execute(text("SELECT 1"))
print("Connected to:", DB_NAME, "@", f"{DB_HOST}:{DB_PORT}")


## 2. Read Every Table

We read all nine tables straight from Postgres into pandas — no filtering yet, just get everything in and look at it.


In [ ]:
TABLES = [
    "olist_customers",
    "olist_orders",
    "olist_order_items",
    "olist_order_payments",
    "olist_order_reviews",
    "olist_products",
    "olist_sellers",
    "olist_geolocation",
    "product_category_translation",
]

raw = {name: pd.read_sql(f'SELECT * FROM "{name}"', engine) for name in TABLES}

for name, df in raw.items():
    print(f"{name:32s} {df.shape[0]:>8,} rows  x  {df.shape[1]:>2} cols")


## 3. Look at Each Table on Its Own

For every table we want to know: **what does one row mean here?** The quickest way to check is comparing the row count to the count of distinct values in what should be the primary key.


In [ ]:
def grain_check(df, key_cols, label):
    n_rows = len(df)
    n_unique_key = df[key_cols].drop_duplicates().shape[0]
    n_dupes = df.duplicated().sum()
    print(f"{label}")
    print(f"  rows                 : {n_rows:,}")
    print(f"  distinct key ({key_cols}): {n_unique_key:,}")
    print(f"  fully duplicated rows: {n_dupes}")
    if n_rows == n_unique_key:
        print("  -> one row per key. Good, this table is already order/entity-level.")
    else:
        print(f"  -> MULTIPLE rows per key ({n_rows - n_unique_key:,} extra). Needs aggregation before any join.")
    print()

grain_check(raw["olist_customers"], ["customer_id"], "olist_customers")
grain_check(raw["olist_orders"], ["order_id"], "olist_orders")
grain_check(raw["olist_products"], ["product_id"], "olist_products")
grain_check(raw["olist_sellers"], ["seller_id"], "olist_sellers")
grain_check(raw["olist_order_items"], ["order_id"], "olist_order_items")
grain_check(raw["olist_order_payments"], ["order_id"], "olist_order_payments")
grain_check(raw["olist_order_reviews"], ["order_id"], "olist_order_reviews")


As expected: `olist_order_items` and `olist_order_payments` have **many rows per `order_id`** — an order can contain several items, and a payment can be split across several transactions (e.g. two credit-card charges, or a voucher plus a card). Both need a `GROUP BY order_id` before they can be joined onto the orders table, otherwise `merge()` would silently duplicate order rows.

`olist_order_reviews` is *almost* one row per order but not quite (a customer can occasionally leave more than one review on the same order). It doesn't matter for us since we're excluding reviews from the ML table anyway.

Quick look at missing keys and order statuses, since not every order will end up usable:


In [ ]:
print("Orders by status:")
print(raw["olist_orders"]["order_status"].value_counts())
print()
print("Orders missing a matching customer_id in olist_customers:",
      (~raw["olist_orders"]["customer_id"].isin(raw["olist_customers"]["customer_id"])).sum())


## 4. Aggregate `order_items` to One Row per Order

Before aggregating, we attach the product's category (translated to English) and physical dimensions to each item row — that way the aggregation carries product information along with it instead of losing it.


In [ ]:
products = raw["olist_products"].merge(
    raw["product_category_translation"], on="product_category_name", how="left"
)

items = raw["olist_order_items"].merge(
    products[[
        "product_id", "product_category_name_english",
        "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm",
    ]],
    on="product_id", how="left",
)

items = items.merge(
    raw["olist_sellers"][["seller_id", "seller_state", "seller_city"]],
    on="seller_id", how="left",
)

items.head(3)


In [ ]:
# One row per order: counts, totals, and a couple of "which one dominates" picks.
items_agg = items.groupby("order_id").agg(
    n_items=("order_item_id", "count"),
    n_distinct_products=("product_id", "nunique"),
    n_distinct_sellers=("seller_id", "nunique"),
    total_price=("price", "sum"),
    total_freight_value=("freight_value", "sum"),
    avg_freight_value=("freight_value", "mean"),
    total_weight_g=("product_weight_g", "sum"),
    max_product_length_cm=("product_length_cm", "max"),
    max_product_height_cm=("product_height_cm", "max"),
    max_product_width_cm=("product_width_cm", "max"),
).reset_index()

# An order can mix categories/sellers. As a simple, defensible rule we tag each
# order with the category and seller of its highest-value item.
idx_top_item = items.groupby("order_id")["price"].idxmax()
top_item = items.loc[idx_top_item, [
    "order_id", "product_category_name_english", "seller_state", "seller_city",
]].rename(columns={
    "product_category_name_english": "main_product_category",
    "seller_state": "main_seller_state",
    "seller_city": "main_seller_city",
})

items_agg = items_agg.merge(top_item, on="order_id", how="left")

print("items_agg:", items_agg.shape, "- one row per order_id:", items_agg["order_id"].is_unique)
items_agg.head(3)


## 5. Aggregate `order_payments` to One Row per Order

Same idea: several payment rows can belong to one order (split payments), so we roll them up first.


In [ ]:
payments = raw["olist_order_payments"]

payments_agg = payments.groupby("order_id").agg(
    total_payment_value=("payment_value", "sum"),
    n_payment_transactions=("payment_sequential", "count"),
    max_payment_installments=("payment_installments", "max"),
).reset_index()

# Tag each order with the payment method used for its largest single transaction.
idx_top_payment = payments.groupby("order_id")["payment_value"].idxmax()
top_payment = payments.loc[idx_top_payment, ["order_id", "payment_type"]].rename(
    columns={"payment_type": "main_payment_type"}
)
payments_agg = payments_agg.merge(top_payment, on="order_id", how="left")

print("payments_agg:", payments_agg.shape, "- one row per order_id:", payments_agg["order_id"].is_unique)
payments_agg.head(3)


## 6. Join Everything into One ML Table

Now that `items_agg` and `payments_agg` are both one-row-per-order, joining them onto `olist_orders` won't multiply any rows. We also bring in the customer's city/state.

Notice we're keeping `order_status`, `order_purchase_timestamp`, `order_approved_at`, `order_delivered_carrier_date`, `order_delivered_customer_date`, and `order_estimated_delivery_date` as **raw columns** — we need them to build the label in Notebook 2. They are **not features yet**, and some of them (like the actual delivery date) never will be — more on that next notebook.


In [ ]:
orders = raw["olist_orders"]
customers = raw["olist_customers"][["customer_id", "customer_city", "customer_state", "customer_zip_code_prefix"]]

ml_table = (
    orders
    .merge(customers, on="customer_id", how="left")
    .merge(items_agg, on="order_id", how="left")
    .merge(payments_agg, on="order_id", how="left")
)

print("ml_table shape:", ml_table.shape)
print("one row per order_id:", ml_table["order_id"].is_unique)
assert ml_table["order_id"].is_unique, "Join produced duplicate orders — check the aggregation step above."
assert len(ml_table) == len(orders), "Row count changed during the join — an aggregation step is missing or wrong."
ml_table.head(3)


## 7. Quick Sanity Check

A few orders should have no items at all (e.g. orders that were never actually processed) — that's expected, not a bug. Let's confirm the pattern makes sense before saving.


In [ ]:
orders_without_items = ml_table["n_items"].isna().sum()
print(f"Orders with no matching order_items row: {orders_without_items:,}")
print()
print("Their order_status breakdown:")
print(ml_table.loc[ml_table["n_items"].isna(), "order_status"].value_counts())


Makes sense — orders with no items are mostly `unavailable` or `canceled`, i.e. orders that never really shipped. We're not dropping anything yet though — filtering happens deliberately in Notebook 2, once we decide what a valid, labelable order actually looks like.

## 8. Save the Artifact


In [ ]:
out_path = ARTIFACTS_DIR / "ml_table.csv"
ml_table.to_csv(out_path, index=False)
print("Saved:", out_path, "-", ml_table.shape)


## Recap

**What's in `ml_table.csv`:**
- One row per `order_id`, matching the original `olist_orders` row count.
- Order metadata: status + all four date columns (purchase, approved, delivered-to-carrier, delivered-to-customer, estimated-delivery).
- Customer city/state.
- Item-level rollups: item/product/seller counts, price & freight totals, weight & size maxima, a "main" product category and seller.
- Payment-level rollups: total paid, number of transactions, installments, "main" payment type.
- **Deliberately excluded:** `olist_order_reviews` (post-delivery, leakage), `olist_geolocation` (many-to-many on ZIP prefix, revisit later if needed for a distance feature).

**Next up — Notebook 2:** turn `order_status` + the two delivery dates into an actual `is_late` label, and check whether we have a class imbalance problem.
